<a href="https://colab.research.google.com/github/ylepen/NPL--M2-DG/blob/main/Text_classification_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Text classification with LLM**

**Downloading the dataset from Hugging Face**

We download the "rotten_tomatoes" dataset

This dataset contains movie reviews. There are:



*   5,331 positive reviews,
*   5,331 negative reviews.

This dataset is a dictionary


In [1]:
from datasets import load_dataset

In [2]:
data = load_dataset("rotten_tomatoes")

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [3]:
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [4]:
data['train'][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

## **Method 1**

We use a **task-specific model** trained for sentiment analysis for tweets

We use the library **transformers**

We create a pipeline: two arguments 1. The task, 2 the model

"text-classification" : The task defining which pipeline will be returned

"text-classification" (alias "sentiment-analysis" available): will return a TextClassificationPipeline.

In [10]:
from transformers import pipeline
pipe = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
import numpy as np
from tqdm import tqdm # to display the iteration
from transformers.pipelines.pt_utils import KeyDataset
# KeyDataset: a utility function from transformer
# Use to efficiently iterate over a single column from a Hugging Fance dataset
# Extract one item at a time

In [12]:
y_pred = []
for result in tqdm(pipe(KeyDataset(data['test'],"text")),
                   total=len(data['test'])):
    # The 'result' from the pipeline for each text input is typically a list
    # containing a single dictionary, e.g., [{'label': 'positive', 'score': 0.95}].

    prediction_data = result[0] if isinstance(result, list) and len(result) > 0 else result # we check if the output is a list or a dictionary
    # usual with Hugging Face pipeline: output can be inconsistent depending on the input

    label = prediction_data.get('label')

    # Map the sentiment label to 0 (negative) or 1 (positive)
    if label == 'positive':
        assignment = 1
    elif label == 'negative':
        assignment = 0
    else:
        # For 'neutral' or any unexpected labels, we'll assign 0 (negative)
        # to fit the binary classification requirement of the dataset.
        assignment = 0
    y_pred.append(assignment)

100%|██████████| 1066/1066 [00:10<00:00, 103.20it/s]


In [13]:
prediction_data

{'label': 'negative', 'score': 0.8362504243850708}

In [8]:
from sklearn.metrics import classification_report
def evaluate_performance(y_true,y_pred):
    performance = classification_report(y_true,y_pred,target_names=["negative","positive"])
    print(performance)

In [ ]:
y_pred

In [9]:
evaluate_performance(data['test']['label'],y_pred)

              precision    recall  f1-score   support

    negative       0.68      0.94      0.79       533
    positive       0.91      0.56      0.69       533

    accuracy                           0.75      1066
   macro avg       0.79      0.75      0.74      1066
weighted avg       0.79      0.75      0.74      1066



**Method 2**

We use a **pretrained embedding model**

We use the package sentence-transformer

all-MiniLM-L6-v2: maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.


In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
train_embeddings = model.encode(data['train']['text'],show_progress_bar=True)
test_embeddings = model.encode(data['test']['text'],show_progress_bar=True)

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [17]:
from sklearn import svm
clf = svm.SVC()
clf.fit(train_embeddings,data['train']['label'])

SVC()

In [18]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data['test']['label'],y_pred)

              precision    recall  f1-score   support

    negative       0.77      0.79      0.78       533
    positive       0.78      0.76      0.77       533

    accuracy                           0.77      1066
   macro avg       0.78      0.77      0.77      1066
weighted avg       0.78      0.77      0.77      1066



## **Text classification without labeled data**

We perform a zero-shot classification:

- we have no labeled data

- we know the definition of the label

- zero-shot classification tries to predict the classification even if we don't have labelled data

# Zero-shot classification with embeddings



1.   We define our labels
2.   We estimate the embedding of the label
3.   We compute a distance metrics (cosine.similarity) between our labels-embeddings and the embeddings of the data (the movie review)
4.   The label with the highest similarity score with the document is selected as selected


In [27]:
label_embeddings = model.encode(["A negative review","A positive review"])

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(test_embeddings,label_embeddings)

y_pred = np.argmax(sim_matrix,axis=1)

In [29]:
evaluate_performance(data['test']['label'],y_pred)

              precision    recall  f1-score   support

    negative       0.73      0.53      0.61       533
    positive       0.63      0.80      0.71       533

    accuracy                           0.67      1066
   macro avg       0.68      0.67      0.66      1066
weighted avg       0.68      0.67      0.66      1066



**Generative** models

Can also be used for zero-shot classification

https://huggingface.co/google/flan-t5-base

In the definition of the pipeline, we select task: "zero-shot-classification"

In [61]:
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm import tqdm

In [62]:
pipe = pipeline(
    task="zero-shot-classification",
    model="google/flan-t5-small",
)

Loading weights:   0%|          | 0/189 [00:00<?, ?it/s]

T5ForSequenceClassification LOAD REPORT from: google/flan-t5-small
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.weight                      | UNEXPECTED | 
classification_head.out_proj.bias   | MISSING    | 
classification_head.out_proj.weight | MISSING    | 
classification_head.dense.weight    | MISSING    | 
classification_head.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


In [59]:

candidate_labels = ["positive", "negative"]

y_pred = []
for result in tqdm(pipe(
    KeyDataset(data["test"], "text"), #KeyDataset(dataset["test"].select(range(100)), "text")
    candidate_labels=candidate_labels,
    batch_size=8
)):
    top_label = result["labels"][0]
    y_pred.append(1 if top_label == "positive" else 0)

1066it [00:11, 93.24it/s] 


In [60]:
evaluate_performance(data['test']['label'],y_pred)

              precision    recall  f1-score   support

    negative       0.78      0.04      0.07       533
    positive       0.51      0.99      0.67       533

    accuracy                           0.51      1066
   macro avg       0.64      0.51      0.37      1066
weighted avg       0.64      0.51      0.37      1066

